# BabyWatcher — Train hand-pose (21 keypoint) cho phân tích hành vi cầm nắm

Notebook này train một model **YOLO-pose 21 keypoint bàn tay** (sơ đồ chuẩn MediaPipe: cổ tay +
4 điểm mỗi ngón × 5 ngón), dùng **dataset chính thức của Ultralytics** (`hand-keypoints.yaml`,
26,768 ảnh, tự động tải) — không cần tự thu thập/gán nhãn dữ liệu tay, và không phụ thuộc model
cộng đồng chưa rõ giấy phép/chất lượng.

Mục tiêu: dùng model này để nhận biết chính xác hành vi **cầm nắm** (khoảng cách đầu ngón cái
so với các đầu ngón khác) — thay cho heuristic thô hiện tại trong `src/detector.py`
(`_detect_hand_closing`, dựa trên tỉ lệ khoảng cách cổ tay/khuỷu tay, không thực sự "nhìn" được
hình dạng bàn tay).

**Trước khi chạy:** `Runtime > Change runtime type > T4 GPU`.

## 0. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 1. Cài đặt thư viện

In [ ]:
!pip install -q ultralytics

## 2. Huấn luyện YOLOv8n-pose trên hand-keypoints — 50 epochs

`data="hand-keypoints.yaml"` là tên config dataset built-in của Ultralytics — thư viện tự tải
dataset (~369MB) trong lần chạy đầu, không cần tải/tải lên thủ công.

Dùng `yolov8n-pose.pt` (nano) làm base — nhẹ, phù hợp chạy song song với `pose_model` (thân
người) và `obj_model` hiện có trong `process_frame()` mà không làm chậm hệ thống quá nhiều.

In [ ]:
import time
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO

model = YOLO("yolov8n-pose.pt")

train_start_time = time.time()
print(f"Bắt đầu huấn luyện lúc: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

results = model.train(
    data="hand-keypoints.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="hand_pose_runs",
    name="detector",
    exist_ok=True,
    save=True,
    plots=True,
    verbose=True,
)

train_end_time = time.time()
elapsed_seconds = train_end_time - train_start_time
elapsed_str = time.strftime("%H:%M:%S", time.gmtime(elapsed_seconds))
print(f"Kết thúc huấn luyện lúc: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tổng thời gian huấn luyện: {elapsed_str} ({elapsed_seconds:.1f}s)")

# Lấy RUN_DIR từ results.save_dir (không hardcode chuỗi) -- bài học từ lần train
# object detector: project/name truyền vào có thể không khớp đường dẫn thật do
# Ultralytics tự thêm tiền tố runs/pose/... tùy phiên bản/cấu hình settings.yaml.
RUN_DIR = str(results.save_dir)
assert (Path(RUN_DIR) / "weights" / "best.pt").exists(), f"Không tìm thấy weights trong {RUN_DIR}"
print("Run directory:", RUN_DIR)

## 3. Precision / Recall / F1 / mAP@50 / mAP@50-95 theo từng epoch (đầu keypoint)

Model pose có **2 đầu ra**: bounding box tay (hậu tố `(B)`) và keypoint (hậu tố `(P)`). Vì mục
tiêu là độ chính xác **vị trí 21 điểm khớp ngón tay**, chỉ số quan trọng nhất là nhóm `(P)`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = [c.strip() for c in df.columns]

precision = df["metrics/precision(P)"]
recall = df["metrics/recall(P)"]
map50 = df["metrics/mAP50(P)"]
map50_95 = df["metrics/mAP50-95(P)"]
f1 = 2 * precision * recall / (precision + recall + 1e-9)
df["f1_pose"] = f1

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], precision, label="Precision (keypoint)")
ax.plot(df["epoch"], recall, label="Recall (keypoint)")
ax.plot(df["epoch"], map50, label="mAP@50 (keypoint)")
ax.plot(df["epoch"], map50_95, label="mAP@50-95 (keypoint)")
ax.plot(df["epoch"], f1, label="F1-score (keypoint)", linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Chỉ số huấn luyện theo epoch — YOLOv8n-pose trên hand-keypoints (50 epochs)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("hand_pose_training_metrics.png", dpi=150)
plt.show()

df[["epoch", "metrics/precision(P)", "metrics/recall(P)", "metrics/mAP50(P)", "metrics/mAP50-95(P)", "f1_pose"]].tail(5)

## 4. Đánh giá cuối cùng trên tập val

Dataset `hand-keypoints.yaml` chỉ có train/val (không có split test riêng), nên đánh giá cuối
chạy trên val — đây cũng là tập `.val()` mặc định dùng khi không truyền `split=`.

In [ ]:
import json

best_pt = Path(RUN_DIR) / "weights" / "best.pt"
best_model = YOLO(str(best_pt))
val_metrics = best_model.val(data="hand-keypoints.yaml")

p = float(val_metrics.pose.mp)
r = float(val_metrics.pose.mr)
map50 = float(val_metrics.pose.map50)
map50_95 = float(val_metrics.pose.map)
f1 = 2 * p * r / (p + r + 1e-9)

print("=== Kết quả keypoint trên tập VAL (best.pt) ===")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1-score:   {f1:.4f}")
print(f"mAP@50:     {map50:.4f}")
print(f"mAP@50-95:  {map50_95:.4f}")
print(f"Thời gian huấn luyện: {elapsed_str} ({elapsed_seconds:.1f}s)")

summary = {
    "pose": {"precision": p, "recall": r, "f1": f1, "map50": map50, "map50_95": map50_95},
    "run_dir": str(RUN_DIR),
    "training_time": {
        "start": datetime.fromtimestamp(train_start_time).strftime("%Y-%m-%d %H:%M:%S"),
        "end": datetime.fromtimestamp(train_end_time).strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_seconds": elapsed_seconds,
        "elapsed_hms": elapsed_str,
    },
}
with open("hand_pose_metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("\nĐã lưu hand_pose_metrics_summary.json")

## 5. Xem các biểu đồ Ultralytics tự sinh

In [ ]:
import os
from IPython.display import Image, display

for fname in ["results.png", "PoseF1_curve.png", "PosePR_curve.png", "PoseP_curve.png", "PoseR_curve.png"]:
    path = f"{RUN_DIR}/{fname}"
    if os.path.exists(path):
        print(fname)
        display(Image(filename=path, width=720))

## 6. Lưu và tải model về

Tải `hand_pose_best.pt` về máy, đặt vào `models/` của đồ án, rồi cập nhật `config.yaml`:

```yaml
models:
  hand_model_path: "models/hand_pose_best.pt"
```

Lưu ý: chỉ đổi `config.yaml` thôi **chưa đủ** — `src/detector.py` hiện đang hardcode
`self.use_hand_detection = False` và `self.hand_detector = None` (phần tích hợp MediaPipe cũ đã
bị gỡ). Cần sửa lại phần load model + logic phân tích cầm nắm trong `detector.py` để dùng model
này — xem trao đổi trong chat để biết chi tiết kiến trúc tích hợp.

In [ ]:
import shutil
from google.colab import files

shutil.copy(f"{RUN_DIR}/weights/best.pt", "hand_pose_best.pt")
files.download("hand_pose_best.pt")
files.download("hand_pose_training_metrics.png")
files.download("hand_pose_metrics_summary.json")